# Step 11 — Accuracy of the classification arms

Everything the paper's accuracy section reports, and the workbook it reports from.

**Arms compared**

| arm | what it is |
|---|---|
| rule engine | notebook 07, register attributes + landuse rules |
| LLM x5 | notebook 10, the same prompt run 5 times per building |
| previous run | the pre-filled labels the validator audited — scores are **definitional**, not an independent estimate |

**Ground truth** is a human audit of the previous run, one colour per dimension:
**green** = that label is correct, **red** = wrong, and for activities the validator
also wrote the corrected set. Bosserhof red rows carry a typed class; activity red
rows carry the corrected label set.

**Two metrics, because the label types differ**

* Bosserhof is one class per building -> plain accuracy.
* Activities are a set per building, and the pipeline redistributes zone weight over
  the set, so a partly-right set is partly useful -> micro precision / recall / F1.

**Population:** buildings with an OSM POI name (reviewable by eye, and the model was
given the names), that carry a decided label. 655 for Bosserhof, 663 for activities.

In [ ]:
import math
import random
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))

from collections import Counter

import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

from config import (LLM_REPRO_COMPARISON, LLM_REPRO_RUNS, VALIDATION_SET_FILE,
                    LLM_REPRO_REVIEW_NEVER, LLM_REPRO_REVIEW_PARTIAL, LLM_REPRO_FINAL,
                    LLM_REPRO_REVIEW_COLOURS, LLM_REPRO_VERDICT_IS_CORRECT,
                    BOSSERHOF_WEIGHTS, arm_comparison)
from validation_utils import (decode_final_validation_set, score_activities,
                              parse_activity_cell)
import openpyxl

pd.set_option('display.width', 240)

BOSS_RUNS = [f'boss_run{i}' for i in range(1, LLM_REPRO_RUNS + 1)]
ACTS_RUNS = [f'acts_run{i}' for i in range(1, LLM_REPRO_RUNS + 1)]
CANON = {str(k).strip().lower() for k in BOSSERHOF_WEIGHTS} | {'(no class)'}


def norm(v):
    import re
    return re.sub(r'\s+', ' ', str(v)).strip().lower()


def to_set(cell):
    """'Leisure; Workers' -> {'Leisure','Workers'}; blank or '(none)' -> empty set."""
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return set()
    text = str(cell).strip()
    if text in ('', '(none)'):
        return set()
    return {p.strip() for p in text.split(';') if p.strip() and p.strip() != '(none)'}


def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return centre - half, centre + half


runs = pd.read_csv(LLM_REPRO_COMPARISON, dtype={'gml_id': str})
rule = pd.read_csv(arm_comparison('rule'), dtype={'gml_id': str})

# Activity truth is re-decoded from the source, never read from the repeat sheet's
# activities_truth column: format_label_set renders both None ("never decided") and
# set() ("residential, no activity here") as an empty string, and the CSV round-trip
# makes both NaN. Those are opposite meanings — the first must be excluded, the second
# is a real answer and the only direct test of over-prediction.
src = decode_final_validation_set(pd.read_csv(VALIDATION_SET_FILE, dtype={'gml_id': str}))
src['gml_id'] = src['gml_id'].astype(str)
src = src.set_index('gml_id')

print(f'{len(runs):,} buildings x {LLM_REPRO_RUNS} runs')

## 1. Population

In [ ]:
df = runs[runs['osm_names'].notna()].merge(
    rule[['gml_id', 'boss_predicted', 'act_predicted', 'boss_human_verdict', 'act_human_verdict']]
    .rename(columns={'boss_human_verdict': 'boss_colour', 'act_human_verdict': 'act_colour'}),
    on='gml_id')
df['acts_truth'] = df['gml_id'].map(src['activities_truth'])
df['acts_previous'] = df['gml_id'].map(src['Predicted_activities']).map(
    lambda x: parse_activity_cell(x)[0] or set())

# Bosserhof: a truth that is not one of the 47 canonical classes cannot be matched by
# anything. 9 rows are multi-label cells the normaliser glued into nonsense, or classes
# outside the vocabulary; excluded rather than scored as automatic failures.
df['boss_valid'] = df['bosserhof_truth'].notna() & df['bosserhof_truth'].map(norm).isin(CANON)
df['acts_valid'] = df['acts_truth'].map(lambda s: isinstance(s, set))

boss = df[df['boss_valid']].reset_index(drop=True)
acts = df[df['bosserhof_truth'].notna() & df['acts_valid']].reset_index(drop=True)

print(f"{len(runs):>4} buildings")
print(f"{int(runs['osm_names'].isna().sum()):>4} dropped, no OSM name")
print(f"{len(df):>4} named")
print(f"{len(boss):>4} scored for Bosserhof   ({len(df) - len(boss)} without a valid single-class truth)")
print(f"{len(acts):>4} scored for activities  ({len(df) - len(acts)} without a decided activity set)")
print()
print('validator colours on the scored rows:')
print(f"  Bosserhof  green {int((boss['boss_colour'] == 'green').sum()):>4}"
      f"  red {int((boss['boss_colour'] == 'red').sum()):>4}")
print(f"  activities green {int((acts['act_colour'] == 'green').sum()):>4}"
      f"  red {int((acts['act_colour'] == 'red').sum()):>4}")

## 2. Bosserhof class — accuracy per arm

One label per building, so this is plain exact-match accuracy with a Wilson interval.
The previous run's row is greyed out conceptually: green *means* it was right, so its
score is fixed by construction and cannot be compared with the others.

In [ ]:
def acc_row(name, ok):
    k, n = int(ok.sum()), len(ok)
    lo, hi = wilson(k, n)
    return {'arm': name, 'correct': k, 'wrong': n - k, 'n': n,
            'accuracy%': round(k / n * 100, 1),
            'ci_low%': round(lo * 100, 1), 'ci_high%': round(hi * 100, 1)}


truth_b = boss['bosserhof_truth'].map(norm)
boss_rows = [acc_row('rule engine', boss['boss_predicted'].map(norm) == truth_b),
             acc_row('LLM majority of 5', boss['boss_modal'].map(norm) == truth_b)]
for c in BOSS_RUNS:
    boss_rows.append(acc_row(f'LLM {c}', boss[c].map(norm) == truth_b))
boss_rows.append(acc_row('previous run (definitional)', boss['boss_colour'] == 'green'))

boss_table = pd.DataFrame(boss_rows).set_index('arm')
print(boss_table.to_string())

llm_runs = boss_table.loc[[f'LLM {c}' for c in BOSS_RUNS], 'accuracy%']
print(f"\nLLM single-run range {llm_runs.min()}-{llm_runs.max()}% "
      f"(SD {llm_runs.std():.2f}); rule engine {boss_table.loc['rule engine', 'accuracy%']}%")

In [ ]:
# Per-building overlap: is the LLM simply better, or better on different buildings?
r_ok = boss['boss_predicted'].map(norm) == truth_b
l_ok = boss['boss_modal'].map(norm) == truth_b
print('rule vs LLM majority, per building:')
print(f'  both right      {int((r_ok & l_ok).sum()):>4}')
print(f'  only LLM right  {int((~r_ok & l_ok).sum()):>4}')
print(f'  only rule right {int((r_ok & ~l_ok).sum()):>4}')
print(f'  both wrong      {int((~r_ok & ~l_ok).sum()):>4}')
print(f"\nthe two arms agree on {(boss['boss_predicted'].map(norm) == boss['boss_modal'].map(norm)).mean():.1%} "
      'of buildings, so they are making different decisions, not variations of one.')

## 3. Bosserhof after the manual review

The green labels were accepted, not re-derived, so every building where the runs did
not reproduce the sheet was reviewed by hand. Both review workbooks encode the verdict
as the fill colour of the cell holding `boss_modal`, i.e. a judgement on the majority
answer. Yellow is dropped for having no verdict; the colour map also accepts blue as
correct, though the reviewer has since resolved those cells to green or red.

In [ ]:
def read_verdicts(path, sheet, verdict_col):
    """Fill colour of `verdict_col` per row, keyed by gml_id — never by row position,
    since a sort in Excel would shift a positional join onto the wrong buildings."""
    ws = openpyxl.load_workbook(path)[sheet]
    header = [c.value for c in ws[1]]
    gid_i, col_i = header.index('gml_id') + 1, header.index(verdict_col) + 1
    out = []
    for r in range(2, ws.max_row + 1):
        gid = ws.cell(r, gid_i).value
        if gid is None:
            continue
        cell = ws.cell(r, col_i)
        rgb = getattr(cell.fill.fgColor, 'rgb', None) if cell.fill.patternType else None
        out.append({'gml_id': str(gid), 'verdict': LLM_REPRO_REVIEW_COLOURS.get(rgb),
                    'reviewed_answer': cell.value})
    frame = pd.DataFrame(out)
    assert frame['verdict'].notna().all(), f'unrecognised fill in {path.name}'
    return frame


reviews = pd.concat([read_verdicts(LLM_REPRO_REVIEW_NEVER, 'Sheet1', 'boss_modal'),
                     read_verdicts(LLM_REPRO_REVIEW_PARTIAL, 'review', 'verdict')],
                    ignore_index=True)
assert reviews['gml_id'].is_unique

full = df[df['bosserhof_truth'].notna()].copy()
full['n_correct'] = [sum(norm(v) == norm(t) for v in r)
                     for r, t in zip(full[BOSS_RUNS].to_numpy(), full['bosserhof_truth'])]
matched = full['n_correct'] == LLM_REPRO_RUNS
# The two workbooks must cover exactly the rows the runs did not reproduce.
assert set(reviews['gml_id']) == set(full.loc[~matched, 'gml_id']), 'review coverage mismatch'

verdict = full['gml_id'].map(reviews.set_index('gml_id')['verdict'])
full['reviewed'] = verdict.notna()
full['outcome'] = verdict.map(LLM_REPRO_VERDICT_IS_CORRECT).map(
    {True: 'correct', False: 'incorrect'})
full.loc[matched, 'outcome'] = 'correct (reproduced the sheet)'
full['outcome'] = full['outcome'].fillna('dropped (reviewer unsure)')

tally = full['outcome'].value_counts()
print(tally.to_string())
scored = full[full['outcome'] != 'dropped (reviewer unsure)']
ok = int(scored['outcome'].str.startswith('correct').sum())
print(f'\nreviewed accuracy: {ok}/{len(scored)} = {ok / len(scored):.1%}')

# The review is one-sided: only rows where the runs disagreed with the sheet were
# examined, so any correction it yields can only help the model. Measure that exposure
# instead of leaving it implicit.
comparable = full[full['reviewed'] &
                  (full['boss_modal'].map(norm) == full['bosserhof_truth'].map(norm)) &
                  (full['outcome'] != 'dropped (reviewer unsure)')]
rejected = int((comparable['outcome'] == 'incorrect').sum())
print(f'\ncaveat: of {len(comparable)} reviewed rows whose majority answer WAS the sheet '
      f'label, the reviewer rejected {rejected} ({rejected / len(comparable):.1%}).')
print(f'the {int(matched.sum())} rows that reproduced the sheet were never reviewed; at that '
      f'rate ~{round(int(matched.sum()) * rejected / len(comparable))} would fail, putting '
      f'accuracy near {(ok - round(int(matched.sum()) * rejected / len(comparable))) / len(scored):.1%}.')

## 4. Activities — precision / recall per arm

In [ ]:
ACTS_TRUTH = list(acts['acts_truth'])
PRED = {c: [to_set(x) for x in acts[c]] for c in ACTS_RUNS}
PRED['rule engine'] = [to_set(x) for x in acts['act_predicted']]
PRED['previous run'] = list(acts['acts_previous'])


def act_metrics(pred, mask=None):
    idx = range(len(acts)) if mask is None else [i for i in range(len(acts)) if mask.iloc[i]]
    m, _ = score_activities([(acts['gml_id'].iloc[i], pred[i], ACTS_TRUTH[i]) for i in idx])
    return m


def act_row(name, m):
    return {'arm': name, 'P%': round(m['precision'] * 100, 1), 'R%': round(m['recall'] * 100, 1),
            'F1%': round(m['f1'] * 100, 1), 'TP': m['n_true_positive'],
            'FP_extra': m['n_over_predicted'], 'FN_missing': m['n_missed'],
            'exact%': round(m['exact_match_rate'] * 100, 1)}


act_rows = [act_row('rule engine', act_metrics(PRED['rule engine']))]
for c in ACTS_RUNS:
    act_rows.append(act_row(f'LLM {c}', act_metrics(PRED[c])))
act_rows.append(act_row('previous run (definitional)', act_metrics(PRED['previous run'])))
act_table = pd.DataFrame(act_rows).set_index('arm')
print(f'{len(acts)} buildings, {sum(len(t) for t in ACTS_TRUTH)} true label instances')
print(act_table.to_string())

llm_f1 = act_table.loc[[f'LLM {c}' for c in ACTS_RUNS], 'F1%']
print(f'\nLLM F1 {llm_f1.mean():.1f}% (SD {llm_f1.std():.2f}, range {llm_f1.max() - llm_f1.min():.2f}pp)'
      f" vs rule engine {act_table.loc['rule engine', 'F1%']}%")

RED = acts['act_colour'] == 'red'
print(f'\non the {int(RED.sum())} red rows — the only ones with an independently written truth:')
for name in ('rule engine', 'acts_run1', 'previous run'):
    key = name if name in PRED else name
    m = act_metrics(PRED[key], RED)
    print(f"  {name:14} F1 {m['f1'] * 100:.1f}%")

## 5. Does the regional activity mix hold up?

The metric that matters for redistribution: how many buildings carry each activity
across the whole region. A per-label bias misallocates that activity's weight however
good the F1 looks, and micro-averaging hides it because frequent labels dominate.

In [ ]:
truth_totals = Counter(l for t in ACTS_TRUTH for l in t)
mix = []
for label in sorted(truth_totals, key=lambda x: -truth_totals[x]):
    llm_counts = [sum(1 for s in PRED[c] if label in s) for c in ACTS_RUNS]
    rule_count = sum(1 for s in PRED['rule engine'] if label in s)
    mix.append({'activity': label, 'truth': truth_totals[label],
                'rule': rule_count,
                'rule_bias%': round((rule_count - truth_totals[label]) / truth_totals[label] * 100, 1),
                'llm_mean': round(sum(llm_counts) / len(llm_counts), 1),
                'llm_bias%': round((sum(llm_counts) / len(llm_counts) - truth_totals[label])
                                   / truth_totals[label] * 100, 1),
                'llm_run_spread': max(llm_counts) - min(llm_counts)})
mix_table = pd.DataFrame(mix)
print(mix_table.to_string(index=False))
print(f"\ntotals: {sum(truth_totals.values())} true vs "
      f"{sum(r['llm_mean'] for r in mix):.0f} LLM / {sum(r['rule'] for r in mix)} rule — "
      'the counts balance, so the error is misallocation BETWEEN activities.')

worst = mix_table.loc[mix_table['llm_bias%'].abs().idxmax(), 'activity']
bad = [i for i in range(len(acts)) if worst in PRED['acts_run1'][i] and worst not in ACTS_TRUTH[i]]
print(f"\n'{worst}' is over-predicted by BOTH arms. On run 1 it is wrong {len(bad)} times "
      f"({len(bad) / act_table.loc['LLM acts_run1', 'FP_extra']:.0%} of all false positives); "
      'those buildings are truly:')
print(pd.Series(Counter(l for i in bad for l in ACTS_TRUTH[i])).sort_values(ascending=False).to_string())

## 6. Reproducibility of the 5 runs

Whether re-running changes the overall picture, compared against the uncertainty the
validation set itself carries.

In [ ]:
random.seed(0)
idx = list(range(len(acts)))
boot = []
for _ in range(2000):
    sample = [random.choice(idx) for _ in idx]
    tp = sum(len(PRED['acts_run1'][i] & ACTS_TRUTH[i]) for i in sample)
    fp = sum(len(PRED['acts_run1'][i] - ACTS_TRUTH[i]) for i in sample)
    fn = sum(len(ACTS_TRUTH[i] - PRED['acts_run1'][i]) for i in sample)
    p, r = tp / (tp + fp), tp / (tp + fn)
    boot.append(200 * p * r / (p + r))
boot.sort()
lo, hi = boot[int(0.025 * len(boot))], boot[int(0.975 * len(boot))]
f1_margin = llm_f1.max() - llm_f1.min()

identical_boss = int((boss[BOSS_RUNS].nunique(axis=1) == 1).sum())
identical_acts = sum(1 for i in range(len(acts))
                     if len({frozenset(PRED[c][i]) for c in ACTS_RUNS}) == 1)
pairs = [(a, b) for ai, a in enumerate(ACTS_RUNS) for b in ACTS_RUNS[ai + 1:]]
jac = sum(sum(len(PRED[a][i] & PRED[b][i]) / len(PRED[a][i] | PRED[b][i])
              if (PRED[a][i] | PRED[b][i]) else 1.0 for a, b in pairs) / len(pairs)
          for i in range(len(acts))) / len(acts)

print(f'Bosserhof: identical class in all {LLM_REPRO_RUNS} runs on '
      f'{identical_boss}/{len(boss)} = {identical_boss / len(boss):.1%}')
print(f'Activities: identical set in all runs on {identical_acts}/{len(acts)} = '
      f'{identical_acts / len(acts):.1%}, mean pairwise Jaccard {jac:.3f}')
print()
print(f'activity F1 run-to-run margin {f1_margin:.2f}pp vs bootstrap 95% CI '
      f'{lo:.1f}-{hi:.1f} (+-{(hi - lo) / 2:.1f}pp) -> sampling error is '
      f'{(hi - lo) / f1_margin:.0f}x wider')
print(f"Bosserhof single-run range {llm_runs.max() - llm_runs.min():.1f}pp")

## 7. The workbook

One file: the conclusions, the per-arm tables, the regional mix, and the per-building
Bosserhof verdicts. Verdict cells are colour-coded; the previous run's rows are marked
as definitional so no reader treats them as a competing score.

In [ ]:
HEAD = (PatternFill('solid', fgColor='FF1F3864'), Font(bold=True, color='FFFFFFFF'))
GOOD = (PatternFill('solid', fgColor='FFC6EFCE'), Font(color='FF006100'))
BAD = (PatternFill('solid', fgColor='FFFFC7CE'), Font(color='FF9C0006'))
WARN = (PatternFill('solid', fgColor='FFFFEB9C'), Font(color='FF9C6500'))
FLAT = (PatternFill('solid', fgColor='FFEDEDED'), Font(color='FF595959'))
TITLE = Font(bold=True, size=14, color='FF1F3864')
LABEL = Font(bold=True, color='FF1F3864')


def dress(ws, widths, freeze='A2'):
    for cell in ws[1]:
        cell.fill, cell.font = HEAD
        cell.alignment = Alignment(vertical='center', wrap_text=True)
    ws.row_dimensions[1].height = 26
    ws.freeze_panes = freeze
    ws.auto_filter.ref = ws.dimensions
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w


def paint(cell, style):
    cell.fill, cell.font = style


conclusions = [
    ('BOSSERHOF CLASS', None, None),
    ('rule engine', boss_table.loc['rule engine', 'accuracy%'] / 100,
     f"{int(boss_table.loc['rule engine', 'correct'])} of {len(boss)} correct"),
    ('LLM, majority of 5', boss_table.loc['LLM majority of 5', 'accuracy%'] / 100,
     f"{int(boss_table.loc['LLM majority of 5', 'correct'])} of {len(boss)} correct"),
    ('LLM, single-run range', None,
     f'{llm_runs.min()}% - {llm_runs.max()}% (SD {llm_runs.std():.2f})'),
    ('after manual review', ok / len(scored), f'{ok} of {len(scored)}; see caveat below'),
    (None, None, None),
    ('ACTIVITY SETS', None, None),
    ('rule engine F1', act_table.loc['rule engine', 'F1%'] / 100,
     f"P {act_table.loc['rule engine', 'P%']}% / R {act_table.loc['rule engine', 'R%']}%"),
    ('LLM F1 (mean of 5)', llm_f1.mean() / 100,
     f"P {act_table.loc[[f'LLM {c}' for c in ACTS_RUNS], 'P%'].mean():.1f}% / "
     f"R {act_table.loc[[f'LLM {c}' for c in ACTS_RUNS], 'R%'].mean():.1f}%"),
    ('difference', (llm_f1.mean() - act_table.loc['rule engine', 'F1%']) / 100,
     'smaller than the run-to-run margin — the arms are tied'),
    (None, None, None),
    ('REPRODUCIBILITY', None, None),
    ('Bosserhof identical in 5 runs', identical_boss / len(boss), 'same class every run'),
    ('activities identical in 5 runs', identical_acts / len(acts),
     f'mean pairwise Jaccard {jac:.3f}'),
    ('activity F1 margin', f1_margin / 100,
     f'vs +-{(hi - lo) / 2:.1f}pp sampling CI — {(hi - lo) / f1_margin:.0f}x wider'),
    (None, None, None),
    ('CAVEATS', None, None),
    ('previous-run scores', None, 'definitional: green MEANS correct, so not comparable'),
    ('review is one-sided', rejected / len(comparable),
     f'reviewer rejected this share of sheet-endorsed labels; the '
     f'{int(matched.sum())} reproduced rows were never reviewed'),
    ('Retail_Daily over-predicted', None,
     'by both arms (+226% LLM, +251% rule) — a label-definition problem, not a model one'),
]

def write_workbook():
    with pd.ExcelWriter(LLM_REPRO_FINAL, engine='openpyxl') as xw:
        boss_table.reset_index().to_excel(xw, sheet_name='bosserhof', index=False)
        act_table.reset_index().to_excel(xw, sheet_name='activities', index=False)
        mix_table.to_excel(xw, sheet_name='regional_mix', index=False)

        buildings = pd.DataFrame({
            'gml_id': full['gml_id'], 'building': full['osm_names'],
            'sheet_label': full['bosserhof_truth'], 'llm_answer': full['boss_modal'],
            'rule_answer': full['gml_id'].map(rule.set_index('gml_id')['boss_predicted']),
            'runs_matching_sheet': full['n_correct'],
            'reviewed_by_hand': full['reviewed'].map({True: 'yes', False: 'no'}),
            'verdict': full['outcome'],
        }).sort_values(['verdict', 'runs_matching_sheet'])
        buildings.to_excel(xw, sheet_name='bosserhof_buildings', index=False)

        # Per-building activity errors, both arms. Sets are rendered as text and the extra
        # / missing columns spell out the diff, so a reader never has to compare two sets
        # by eye to see what went wrong.
        def direction(pred, truth):
            extra, missing = pred - truth, truth - pred
            if not extra and not missing:
                return 'exact'
            if extra and missing:
                return 'both'
            return 'extra only' if extra else 'missing only'


        def render(labels):
            return '; '.join(sorted(labels)) if labels else '(none)'


        runs_exact = [sum(1 for c in ACTS_RUNS if PRED[c][i] == ACTS_TRUTH[i])
                      for i in range(len(acts))]
        # Labels the runs disagree about — present in some runs but not all. This is what
        # the 5 repeats add over a single run, so it gets its own column rather than being
        # left for the reader to spot by comparing five set columns.
        unstable = [render(set().union(*(PRED[c][i] for c in ACTS_RUNS)) -
                           set.intersection(*(PRED[c][i] for c in ACTS_RUNS)))
                    for i in range(len(acts))]

        act_buildings = pd.DataFrame({
            'gml_id': acts['gml_id'],
            'building': acts['osm_names'],
            'truth': [render(t) for t in ACTS_TRUTH],
            **{f'llm_run{n}': [render(s) for s in PRED[f'acts_run{n}']]
               for n in range(1, LLM_REPRO_RUNS + 1)},
            'unstable_labels': unstable,
            'runs_exact_of_5': runs_exact,
            # The diff is spelled out against run 1 only — one representative run, so the
            # column headings cannot be misread as applying to all five.
            'run1_extra': [render(PRED['acts_run1'][i] - ACTS_TRUTH[i]) for i in range(len(acts))],
            'run1_missing': [render(ACTS_TRUTH[i] - PRED['acts_run1'][i]) for i in range(len(acts))],
            'run1_result': [direction(PRED['acts_run1'][i], ACTS_TRUTH[i]) for i in range(len(acts))],
            'rule_answer': [render(s) for s in PRED['rule engine']],
            'rule_result': [direction(PRED['rule engine'][i], ACTS_TRUTH[i]) for i in range(len(acts))],
            'validator': acts['act_colour'],
        }).sort_values(['run1_result', 'runs_exact_of_5'])
        act_buildings.to_excel(xw, sheet_name='activity_buildings', index=False)

        wb = xw.book
        ws = wb.create_sheet('conclusions', 0)
        ws['A1'] = 'Classification accuracy — rule engine vs LLM (5 runs)'
        ws['A1'].font = TITLE
        ws['A2'] = f'{len(boss)} buildings for Bosserhof, {len(acts)} for activities. ' \
                   'Truth from a human audit of the previous run.'
        ws['A2'].font = Font(italic=True, color='FF595959')
        r = 4
        at = {}                     # label -> row, so highlighting never hardcodes a row
        for label, value, note in conclusions:
            if label is None:
                r += 1
                continue
            at[label] = r
            cell = ws.cell(r, 1, label)
            if value is None and note is None:
                cell.font = Font(bold=True, color='FFFFFFFF')
                cell.fill = PatternFill('solid', fgColor='FF1F3864')
            else:
                cell.font = LABEL
            if value is not None:
                v = ws.cell(r, 2, value)
                v.number_format = '0.0%'
                v.alignment = Alignment(horizontal='right')
            if note:
                ws.cell(r, 3, note).font = Font(color='FF595959', size=10)
            r += 1
        for col, w in zip('ABC', (30, 12, 70)):
            ws.column_dimensions[col].width = w

        # Highlight the comparisons a reader should land on first. Looked up by label
        # because an inserted line would otherwise shift every hardcoded cell reference
        # and silently colour the wrong number.
        paint(ws.cell(at['rule engine'], 2), BAD)
        paint(ws.cell(at['LLM, majority of 5'], 2), GOOD)
        paint(ws.cell(at['after manual review'], 2), GOOD)
        paint(ws.cell(at['difference'], 2), WARN)

        dress(wb['bosserhof'], (28, 9, 8, 7, 11, 9, 10))
        dress(wb['activities'], (28, 8, 8, 8, 8, 10, 11, 9))
        dress(wb['regional_mix'], (20, 8, 8, 11, 10, 11, 14))
        dress(wb['bosserhof_buildings'], (10, 30, 26, 26, 26, 10, 10, 30))
        dress(wb['activity_buildings'],
              (10, 26, 30) + (30,) * LLM_REPRO_RUNS + (22, 9, 18, 18, 13, 30, 13, 10))

        for name, col in (('bosserhof', 'arm'), ('activities', 'arm')):
            ws2 = wb[name]
            idx_col = 1
            for row in range(2, ws2.max_row + 1):
                if 'definitional' in str(ws2.cell(row, idx_col).value):
                    for c in ws2[row]:
                        paint(c, FLAT)
        ws2 = wb['bosserhof_buildings']
        vcol = list(buildings.columns).index('verdict') + 1
        for row in range(2, ws2.max_row + 1):
            v = str(ws2.cell(row, vcol).value)
            paint(ws2.cell(row, vcol),
                  GOOD if v.startswith('correct') else BAD if v.startswith('incorrect') else WARN)
        ws2 = wb['regional_mix']
        for row in range(2, ws2.max_row + 1):
            for col in (4, 6):
                cell = ws2.cell(row, col)
                paint(cell, BAD if abs(cell.value) > 25 else
                      WARN if abs(cell.value) > 10 else GOOD)

        # Activity errors: exact is green, a one-directional error amber, a swap red — the
        # same reading as the direction table, so the sheet and the summary agree.
        ws2 = wb['activity_buildings']
        cols = list(act_buildings.columns)
        style_for = {'exact': GOOD, 'extra only': WARN, 'missing only': WARN, 'both': BAD}
        for row in range(2, ws2.max_row + 1):
            for name in ('run1_result', 'rule_result'):
                cell = ws2.cell(row, cols.index(name) + 1)
                paint(cell, style_for[str(cell.value)])
                cell.alignment = Alignment(horizontal='center')
            n_exact = ws2.cell(row, cols.index('runs_exact_of_5') + 1)
            paint(n_exact, GOOD if n_exact.value == LLM_REPRO_RUNS
                  else BAD if n_exact.value == 0 else WARN)
            n_exact.alignment = Alignment(horizontal='center')

    return wb

# Excel keeps an exclusive lock on an open file, and this workbook is meant to be
# read by eye — so it will often BE open when the notebook is re-run. Report that
# plainly instead of dying on a raw PermissionError, and never claim a write that
# did not happen: a stale workbook on disk looks exactly like a freshly built one.
try:
    wb = write_workbook()
    written = True
except PermissionError:
    written = False
    print(f'NOT WRITTEN — {LLM_REPRO_FINAL.name} is open in Excel.')
    print('close it and re-run this cell; the file on disk is from an earlier run.')

if written:
    print(f'{LLM_REPRO_FINAL}')
    for name in wb.sheetnames:
        print(f'  {name:20} {wb[name].max_row - 1:>4} rows')

In [ ]:
# Re-derive the headline from the written file: the workbook a reader opens must carry
# the same numbers this notebook printed.
if not written:
    print('skipped — this run did not write the workbook (it was open in Excel)')
    raise SystemExit

check = pd.read_excel(LLM_REPRO_FINAL, sheet_name='bosserhof').set_index('arm')
assert check.loc['rule engine', 'accuracy%'] == boss_table.loc['rule engine', 'accuracy%']
assert check.loc['LLM majority of 5', 'accuracy%'] == boss_table.loc['LLM majority of 5', 'accuracy%']
chk_b = pd.read_excel(LLM_REPRO_FINAL, sheet_name='bosserhof_buildings')
chk_a = pd.read_excel(LLM_REPRO_FINAL, sheet_name='activity_buildings')
assert len(chk_b) == len(full) and len(chk_a) == len(acts)
# every building must carry a direction for both arms, or the sheet has a blind spot
assert chk_a['run1_result'].notna().all() and chk_a['rule_result'].notna().all()
assert all(f'llm_run{n}' in chk_a.columns for n in range(1, LLM_REPRO_RUNS + 1)),     'every run must have its own column'
print('verified from the file:')
print(f"  Bosserhof  rule {check.loc['rule engine', 'accuracy%']}% vs "
      f"LLM {check.loc['LLM majority of 5', 'accuracy%']}%")
print(f"  activities rule {act_table.loc['rule engine', 'F1%']}% vs LLM {llm_f1.mean():.1f}% F1")
print(f'  {len(chk_b)} Bosserhof rows, {len(chk_a)} activity rows')
print('  activity results (run 1):',
      {k: int(v) for k, v in chk_a['run1_result'].value_counts().items()})
print(f"  buildings where the runs disagree on at least one label: "
      f"{int((chk_a['unstable_labels'] != '(none)').sum())}")